In [ ]:
import numpy as np
import math
import time
import sys
import pandas as pd
import itertools
import copy
from scipy.stats import binom
from scipy.stats import rankdata
from math import sqrt

In [ ]:
Num_Iteration = 200  # Generations

Num_Chrom = 100      # Population size
Num_n = 8            # nodes
Num_m = 2            # paths 
Demand = 30

##############################
Num_Archive = 100
Archive = []

In [ ]:
# Read NSGA-II(new).xlsx
Capacity_prob_table_read = pd.read_excel('NSGA-II(new).xlsx', sheet_name = 'Prob. Table', header=[0, 1])
# Read AMC_Matrix.xlsx
Transition_Probability_New_Matrix_read = pd.read_excel('AMC_Matrix.xlsx',sheet_name = 'New', header=0 , index_col=0) 
Transition_Probability_Rema_Matrix_read = pd.read_excel('AMC_Matrix.xlsx',sheet_name = 'Remanu.', header=0 , index_col=0) 
Workstation_States_read = pd.read_excel('AMC_Matrix.xlsx', sheet_name = 'Workstation_States', header = [0,1], index_col=0)

In [ ]:
def initpop():
    New_Max_Num = Capacity_prob_table_read['New']['Max_num']
    Rema_Max_Num = Capacity_prob_table_read['Rema.']['Max_num']
    Population = []
    Population_Num = []
    for i in range(Num_Chrom):
        Chromosome = np.random.randint(2,size = (Num_m, Num_n))
        Chromosome_Num = np.zeros((Num_m, Num_n))
        for j in range(Num_m):
            for i in range(Num_n):
                if Chromosome[j][i] == 0: 
                    Chromosome_Num[j][i] = np.random.randint(1, New_Max_Num[i]+1)
                else:
                    Chromosome_Num[j][i] = np.random.randint(1, Rema_Max_Num[i]+1) 
        Population.append(Chromosome)
        Population_Num.append(Chromosome_Num)
    return Population, Population_Num, New_Max_Num, Rema_Max_Num

Population, Population_Num, New_Max_Num, Rema_Max_Num = initpop() #0=new, 1=remanu.
np.save("Population.npy", Population)
np.save("Population_Num.npy", Population_Num)
np.save("Max_Num.npy", {"New_Max_Num": New_Max_Num.values, "Rema_Max_Num": Rema_Max_Num.values})


In [ ]:
def AMC_Markov_Transition_Probability_Read():
    #讀取Transition_Probability_Matrix
    Transition_Probability_New_Matrix = Transition_Probability_New_Matrix_read.to_numpy()
    Transition_Probability_Rema_Matrix = Transition_Probability_Rema_Matrix_read.to_numpy()    
    
    return Transition_Probability_New_Matrix, Transition_Probability_Rema_Matrix

Transition_Probability_New_Matrix, Transition_Probability_Rema_Matrix = AMC_Markov_Transition_Probability_Read()

In [ ]:
def AMC_Markov_Reconstruction_Transition_Probability():   
    Result_Reconstruction_Matrix = []
    
    for row in range(Num_m):
        Reconstruction_transition_matrix = np.zeros_like(Transition_Probability_New_Matrix)
        for i in range(len(Chromosome[row])):
            if Chromosome[row][i] == 0:
                Reconstruction_transition_matrix[i] = Transition_Probability_New_Matrix[i]
            else:
                Reconstruction_transition_matrix[i] = Transition_Probability_Rema_Matrix[i]

                
        #State_reworking的workstation重建
        #states_9 = 3
        #states_10 = 4
        #states_11 = 5 
        states = {8: (3, 2), 9: (4, 3), 10: (5, 4)}
        for Rework_state, (state_index, chromosome_index) in states.items():
            if Chromosome[row][chromosome_index] == 0:
                Reconstruction_transition_matrix[Rework_state] = Transition_Probability_New_Matrix[Rework_state]
            else:
                Reconstruction_transition_matrix[Rework_state] = Transition_Probability_Rema_Matrix[Rework_state]
        
        max_state_index = max(state_index for state_index, (_, _) in states.items())
        for z in range(max_state_index+1, max_state_index+3):
            Reconstruction_transition_matrix[z] = Transition_Probability_New_Matrix[z]
            
        Result_Reconstruction_Matrix.append(Reconstruction_transition_matrix)
    return Result_Reconstruction_Matrix

In [ ]:
def AMC_Markov_Calculate(Result_Reconstruction_Matrix):
    Output_prob = []
    E_First_row = []
    
    #計算Expected_Time_Matrix
    for Reconstruction_transition_matrix in Result_Reconstruction_Matrix:    
        n = Reconstruction_transition_matrix.shape[0]  #找出Q的列、欄數
        Q = Reconstruction_transition_matrix[:n-2, :n-2]
        I = np.eye(n-2) #建構單位矩陣(I=(n-2)*(n-2))
        E = np.linalg.inv(I - Q)
        E_First_row.append(E[0])
        R = Reconstruction_transition_matrix[:n-2, n-2:]
        A = E[0, :] @ R[:, -1]
        Output_prob.append(A)
    return np.array(Output_prob), np.array(E_First_row)

In [ ]:
def Enumerant_F():
    Ranges = [range(Demand + 1)] * Num_m
    F_vectors = [combo for combo in itertools.product(*Ranges) if sum(combo) == Demand]
    return F_vectors

In [ ]:
def Calculate_Input():
    Input = []
    for F in F_vectors:
        Input.append([f / output for f, output in zip(F, Output_prob)]) 
    return Input

In [ ]:
def Max_capacity_matrix_New_Remanu(Capacity_prob_table_read, num_m, num_n):  #找出vertex對應New和Remanu.最大的capacity
    Vertices = Capacity_prob_table_read[('workstation','vertex')].fillna(method='ffill')  #將NaN填為前一個vertex值
    Unique_vertices = Vertices.unique()  #找出所有的vertex
    Max_capacity_matrix = np.zeros((2, num_n), dtype=int)  #初始化2*num_n的矩陣，設為整數(new和remanu.最大capacity的矩陣)
    Vertex_index = 0  #紀錄當前處理的vertex在矩陣中的位置
    for Vertex in Unique_vertices:
        Vertex_data = Capacity_prob_table_read[Vertices == Vertex]
        Max_new_capacity = Vertex_data[('New', 'capacity')].max()
        Max_remanu_capacity = Vertex_data[('Rema.', 'capacity')].max()
        if pd.isna(Max_new_capacity) or pd.isna(Max_remanu_capacity):
            continue  #如果capacity為NaN，跳過
        #將數據填入對應的矩陣中
        j = Vertex_index  #這裡只需要列的索引
        if j < num_n:  #確保不會超出矩陣大小
            Max_capacity_matrix[0, j] = int(Max_new_capacity)  #第一列存 New 的最大 capacity
            Max_capacity_matrix[1, j] = int(Max_remanu_capacity)  #第二列存 Remanu. 的最大 capacity
        Vertex_index += 1
    
    return Max_capacity_matrix

Max_capacity_matrix = Max_capacity_matrix_New_Remanu(Capacity_prob_table_read, Num_m, Num_n)

In [ ]:
def Find_max_capacity_from_chromosome(Chromosome):
    num_m, num_n = Chromosome.shape
    Find_max_capacity = np.zeros((num_m, num_n))  # 初始化 Num_m * Num_n 的結果矩陣
    for i in range(num_m):
        for j in range(num_n):
            # 如果 Chromosome[i, j] == 1，選擇 capacity_matrix[0, j]；否則選擇 capacity_matrix[1, j]
            Find_max_capacity[i, j] = Max_capacity_matrix[Chromosome[i, j], j]
    return Find_max_capacity

In [ ]:
def Find_Lj_Chromosome(Find_max_capacity, Chromosome_num):
    Lj = [ ]
    for i in range(Num_m):
        min_value = int(np.min(Find_max_capacity[i]*Chromosome_num[i]))   # 找第 i 列中的最小值
        Lj.append(min_value)                       # 將最小值加入 Lj
    return Lj

In [ ]:
def Result_Input_Capacity_Chromosome(Input, Lj):
    Result_Input_Capacity = []
    for Combination in Input:
        if all((Combination)[i] <= Lj[i] for i in range(len(Lj))):    # 將元組轉為列表進行比較
            Result_Input_Capacity.append(Combination)
    return Result_Input_Capacity

In [ ]:
def Flow_Workstation(Result_Input_Capacity, E_First_row, Find_max_capacity, Chromosome_num):
    Workstation_States_read = pd.read_excel('AMC_Matrix.xlsx', sheet_name = 'Workstation_States', header = [0,1], index_col=0)
    Workstation_States = Workstation_States_read.to_numpy()
    Result_Workstation_Input = []
    for z in Result_Input_Capacity:
        Each_Workstation_Input = np.zeros((Num_m, Num_n), dtype=float)
        all_valid = True #假設所有j符合條件
        for j in range(Num_m):
            Workstation_Input = np.zeros(Workstation_States.shape[1], dtype=float)
            for i in range(Num_n):
                Workstation_States_Position = np.where(Workstation_States[:,i] == 1)[0]
                Workstation_Input[i] = z[j] * np.sum(E_First_row[j][Workstation_States_Position]) 
                Each_Workstation_Input[j][i] = Workstation_Input[i]
            if not np.all(Each_Workstation_Input[j] <= Find_max_capacity[j, :]*Chromosome_num[j, :]):
                all_valid = False  #如果有不符合的，則標記為 False
                break  #退出檢查
        if all_valid:
            Result_Workstation_Input.append(Each_Workstation_Input)
    return np.array(Result_Workstation_Input)

In [ ]:
def Calculate_Min_num_machine(Result_Workstation_Input, Find_max_capacity, Chromosome):
    Min_num_machine = []
    for s in range(len(Result_Workstation_Input)):
        Each_Workstation_machine = np.zeros((Num_m, Num_n), dtype=int)
        for j in range(Num_m):
            for i in range(Num_n):
                Each_Workstation_machine[j][i] = math.ceil(Result_Workstation_Input[s][j][i]/Find_max_capacity[j][i])
        Min_num_machine.append(Each_Workstation_machine)
    return Min_num_machine

In [ ]:
def Read_Energy_Data():
    Read_Energy_Machine = np.zeros((2, Num_n)) #2: rema. or new
    Read_Price_Machine = np.zeros((2, Num_n))
    Read_Production = np.zeros((2, Num_n))
    Read_Distance = np.zeros((2, Num_n))
    Read_Emission_Factor = np.zeros((2, Num_n))

    for j in range(2):
        for i in range(Num_n):
            vertex_name = f'v1,{i + 1}'
            if j == 0:
                Read_Energy_Machine[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('New', 'Energy_Machine(kW)')].dropna().values
                Read_Price_Machine[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('New', 'Price_Machine(US)')].dropna().values
                Read_Production[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('New', 'Total_Production')].dropna().values
                Read_Distance[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('New', 'Distance(m)')].dropna().values
                Read_Emission_Factor[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('New', 'emission factor')].dropna().values 
            else: 
                Read_Energy_Machine[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('Rema.', 'Energy_Machine(kW)')].dropna().values
                Read_Price_Machine[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('Rema.', 'Price_Machine(US)')].dropna().values
                Read_Production[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('Rema.', 'Total_Production')].dropna().values
                Read_Distance[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('Rema.', 'Distance(m)')].dropna().values
                Read_Emission_Factor[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('Rema.', 'emission factor')].dropna().values                    
    return Read_Energy_Machine, Read_Price_Machine, Read_Production, Read_Distance, Read_Emission_Factor

Read_Energy_Machine, Read_Price_Machine, Read_Production, Read_Distance, Read_Emission_Factor = Read_Energy_Data()

In [ ]:
def Energy_Data(Chromosome):
    Energy_Machine = np.zeros((Num_m, Num_n)) 
    Price_Machine = np.zeros((Num_m, Num_n))
    Production = np.zeros((Num_m, Num_n))
    Distance = np.zeros((Num_m, Num_n))
    Emission_Factor = np.zeros((Num_m, Num_n))
    
    for j in range(Num_m):
        for i in range(Num_n):
            if Chromosome[j][i] == 0:
                Energy_Machine[j][i] = Read_Energy_Machine[0][i]
                Price_Machine[j][i] = Read_Price_Machine[0][i]
                Production[j][i] = Read_Production[0][i]
                Emission_Factor[j][i] = Read_Emission_Factor[0][i] 
            else:
                Energy_Machine[j][i] = Read_Energy_Machine[1][i]
                Price_Machine[j][i] = Read_Price_Machine[1][i]
                Production[j][i] = Read_Production[1][i]
                Emission_Factor[j][i] = Read_Emission_Factor[1][i] 
    return Energy_Machine, Price_Machine, Production, Distance, Emission_Factor

In [ ]:
def Calculate_Result_F_Energy_Scope1_2(Min_num_machine, Find_max_capacity, Energy_Machine, Distance):
    Mu = 0.3       #摩擦係數
    m = 0.074      #一片PCB的重量(kg)
    g = 9.81       #重力加速度
    EFa = 0.509    #kgCo2e/kWh
    Ea = 100000000    #Scope 1&2
    Energy_Time = 1   #機台運作1hr
    Result_F_Energy_Scope1_2 = []
    for Combination in Min_num_machine:
        Energy_Consumption = 0
        for j in range(Num_m):
            for i in range(Num_n):
                Energy_Consumption += ((Energy_Machine[j][i] * Energy_Time) * Combination[j][i]  + (Mu * (m * Find_max_capacity[j][i]*100) * g * Distance[j][i]) * Combination[j][i]) * EFa 
        if Energy_Consumption <= Ea:
            Result_F_Energy_Scope1_2.append(Combination)
            
    
    return Result_F_Energy_Scope1_2

In [ ]:
def Calculate_Result_F_Energy_Scope3(Result_F_Energy_Scope1_2, Find_max_capacity, Price_Machine, Emission_Factor, Production):
    Eb = 55000      #Scope 3
    Result_F_Energy_Scope3 = []
    for Combination in Result_F_Energy_Scope1_2:
        Energy_Consumption = 0
        for j in range(Num_m):
            for i in range(Num_n):
                Energy_Consumption += (Price_Machine[j][i] * 10000) * Combination[j][i] * (Find_max_capacity[j][i] * Combination[j][i] / Production[j][i] * Combination[j][i]) * Emission_Factor[j][i] 
        if Energy_Consumption <= Eb:
            Result_F_Energy_Scope3.append(Combination)
        
    F_Energy_Scope3 = []
    for item in Result_F_Energy_Scope3:
        if tuple(map(tuple, item)) not in F_Energy_Scope3:
            F_Energy_Scope3.append(tuple(map(tuple, item)))
    return [list(map(list, comb)) for comb in F_Energy_Scope3]            

In [ ]:
def Calculate_d_MP_Candidates(Result_F_Energy_Scope3, Chromosome, Find_max_capacity):
    d_MP_Candidates = []
    for z in range(len(Result_F_Energy_Scope3)):
        Row_results = [] #每個Production Line
        for j in range(Num_m):
            Column_results = []  #每個Workstation
            for i in range(Num_n):
                Capacities = Find_max_capacity[j][i] * Result_F_Energy_Scope3[z][j][i]
                Column_results.append(Capacities) 
            Row_results.append(Column_results) 
        d_MP_Candidates.append(Row_results)
    return d_MP_Candidates

In [ ]:
def Calculate_d_MP(d_MP_Candidates, Result_F_Energy_Scope3):
    indices_to_keep = list(range(len(d_MP_Candidates)))  
    
    for i in range(len(d_MP_Candidates)):
        for j in range(len(d_MP_Candidates)):
            if i != j:
                x = d_MP_Candidates[i] 
                y = d_MP_Candidates[j]
                condition1 = all(x_i >= y_i for x_i, y_i in zip(x, y))
                condition2 = any(x_i > y_i for x_i, y_i in zip(x, y))
                if condition1 and condition2:
                    if i in indices_to_keep:  
                        indices_to_keep.remove(i)
                    break
                
    d_MP = [d_MP_Candidates[idx] for idx in indices_to_keep]
    corresponding_Result_F_Energy_Scope3 = [Result_F_Energy_Scope3[idx] for idx in indices_to_keep]

    return d_MP, corresponding_Result_F_Energy_Scope3

In [ ]:
def Load_Prob_Data (Chromosome):
    Prob_workstation = np.zeros((2, Num_n))
    for j in range(2):
        for i in range(Num_n):
            vertex_name = f'v1,{i + 1}'
            if Chromosome[j][i] == 0:
                Prob_workstation[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('New', 'operation prob.')].dropna().values
            else: 
                Prob_workstation[j][i] = Capacity_prob_table_read.loc[Capacity_prob_table_read[('workstation','vertex')] == vertex_name,('Rema.', 'operation prob.')].dropna().values                   
    return Prob_workstation

In [ ]:
def Calculate_System_Reliability(d_MP, Num_z, Prob_workstation, Chromosome_num):
    if len(d_MP)!=0:
        Reliability = 1
        for j in range(Num_m):
            for i in range(Num_n):
                vertex_name = f'v1,{i + 1}'
                prob_sum = sum(binom.pmf(x, Chromosome_num[j][i], Prob_workstation[j][i]) for x in range(d_MP[0][j][i], int(Chromosome_num[j][i])+1))
                Reliability *= prob_sum
        
        if Num_z > 1:
            for z in range(1, Num_z):  # 第2個 d_MP 開始
                Temp_1 = 1
                for j in range(Num_m):
                    for i in range(Num_n):
                        vertex_name = f'v1,{i + 1}'
                        prob_sum = sum(binom.pmf(x, Chromosome_num[j][i], Prob_workstation[j][i]) for x in range(d_MP[z][j][i], int(Chromosome_num[j][i])+1))  
                        Temp_1 *= prob_sum
                Temp_Y_list = []
                for r in range(z):
                    Temp_Y = d_MP[z].copy()
                    for j in range(Num_m):
                        for i in range(Num_n):
                            Temp_Y[j][i] = max(Temp_Y[j][i], d_MP[r][j][i])
                    Temp_Y_list.append(Temp_Y)
                if len(Temp_Y_list) == 1:
                    Temp_2 = 1
                    for j in range(Num_m):
                        for i in range(Num_n):
                            prob_sum = sum(binom.pmf(x, Chromosome_num[j][i], Prob_workstation[j][i]) for x in range(Temp_Y_list[0][j][i], int(Chromosome_num[j][i])+1))
                            Temp_2 *= prob_sum
                else:
                    filtered_list = filter_Temp_Y_list(Temp_Y_list)
                    Temp_2 = Calculate_System_Reliability(filtered_list, len(filtered_list), Prob_workstation, Chromosome_num)
                R_Temp = Temp_1 - Temp_2
    
                Reliability += R_Temp
    else:
        Reliability = 0
    return Reliability

def filter_Temp_Y_list(Temp_Y_list):
    filtered_list = []
    for i in range(len(Temp_Y_list)):
        keep_item = True
        for j in range(len(Temp_Y_list)):
            if i != j:
                x = Temp_Y_list[i]
                y = Temp_Y_list[j]
                condition1 = True
                condition2 = False
                for row_x, row_y in zip(x, y):
                    row_condition1 = all(x_i >= y_i for x_i, y_i in zip(row_x, row_y))  
                    row_condition2 = any(x_i > y_i for x_i, y_i in zip(row_x, row_y))   

                    condition1 = condition1 and row_condition1 
                    condition2 = condition2 or row_condition2  
                if condition1 and condition2:
                    keep_item = False
                    break
        if keep_item:
            filtered_list.append(Temp_Y_list[i]) 
    return filtered_list

In [ ]:
def Calculate_Cost_Machine(Price_Machine, Chromosome_num):
    Cost_Machine = 0
    for j in range(Num_m):
        for i in range(Num_n):
            Cost_Machine += (Price_Machine[j][i]*10000) * Chromosome_num[j][i]
    return Cost_Machine

In [ ]:
S = []
C = []
for Chromosome, Chromosome_num in zip(Population, Population_Num):
    Result_Reconstruction_Matrix = AMC_Markov_Reconstruction_Transition_Probability()
    Output_prob, E_First_row = AMC_Markov_Calculate(Result_Reconstruction_Matrix)
    F_vectors = Enumerant_F()
    Input = Calculate_Input()
    Find_max_capacity = Find_max_capacity_from_chromosome(Chromosome)
    Lj = Find_Lj_Chromosome(Find_max_capacity, Chromosome_num)
    Result_Input_Capacity = Result_Input_Capacity_Chromosome(Input, Lj)
    Result_Workstation_Input = Flow_Workstation (Result_Input_Capacity, E_First_row, Find_max_capacity, Chromosome_num)
    Min_num_machine = Calculate_Min_num_machine(Result_Workstation_Input, Find_max_capacity, Chromosome)
    Energy_Machine, Price_Machine, Production, Distance, Emission_Factor = Energy_Data(Chromosome)
    Result_F_Energy_Scope1_2 = Calculate_Result_F_Energy_Scope1_2(Min_num_machine, Find_max_capacity, Energy_Machine, Distance)
    Result_F_Energy_Scope3 = Calculate_Result_F_Energy_Scope3(Result_F_Energy_Scope1_2, Find_max_capacity, Price_Machine, Emission_Factor, Production)
    d_MP_Candidates = Calculate_d_MP_Candidates(Result_F_Energy_Scope3, Chromosome, Find_max_capacity)
    d_MP, corresponding_Result_F_Energy_Scope3 = Calculate_d_MP(d_MP_Candidates, Result_F_Energy_Scope3)
    Prob_workstation = Load_Prob_Data (Chromosome)
    d_MP = np.array(corresponding_Result_F_Energy_Scope3)  # 將 d_MP 轉為陣列型式(機台數量)
    Num_z = d_MP.shape[0]
    System_Reliability = Calculate_System_Reliability(d_MP, Num_z, Prob_workstation, Chromosome_num)
    S.append(System_Reliability)
    Cost = Calculate_Cost_Machine(Price_Machine, Chromosome_num)
    C.append(Cost)

In [ ]:
def strength(S, C, Population):  #第1個generation沒有archive
    strength_value = [0] * (len(Population))
    index = [[] for _ in range(len(Population))]
    for p in range(len(Population)):
        for q in range(len(Population)):
            if p != q:
                if C[p] < C[q] and S[p] > S[q]:
                    strength_value[p] += 1
                elif C[p] == C[q] and S[p] > S[q]:
                    strength_value[p] += 1                
                elif C[p] < C[q] and S[p] == S[q]: 
                    strength_value[p] += 1
                elif C[p] > C[q] and S[p] < S[q]:
                    index[p].append(q)           
                elif C[p] == C[q] and S[p] < S[q]:
                    index[p].append(q)             
                elif C[p] > C[q] and S[p] == S[q]:
                    index[p].append(q)
    return strength_value, index

strength_value, index = strength(S, C, Population)

In [ ]:
def Raw_Fitness_Calculate(strength_value, index, Population):
    Raw_Fitness = [0] * (len(Population))
    for p in range(len(Population)):
        Fitness = 0
        for q in index[p]:
            Fitness += strength_value[q]
        Raw_Fitness[p] = Fitness
    return Raw_Fitness

Raw_Fitness = Raw_Fitness_Calculate(strength_value, index, Population)

In [ ]:
def Density_Calculate(S, C, Population): 
    k = math.floor((Num_Chrom + Num_Archive)**(1/2))
    Density = [0] * (len(Population))
    for p in range(len(Population)):
        Distance = []        
        for q in range(len(Population)):
            Distance_Calculate = 0
            if p != q:
                Distance_Calculate = ((S[p] - S[q]) ** 2 + (C[p] - C[q]) ** 2)**(1/2)
                Distance.append(Distance_Calculate)
        Distance.sort()
        Density[p] = 1/(Distance[k-1]+2)             
    return Density
                       
Density = Density_Calculate(S, C, Population)

In [ ]:
def Fitness_Calculate(Raw_Fitness, Density, Population):
    Fitness =[0] * (len(Population))
    for p in range(len(Population)):
        Fitness[p] = Raw_Fitness[p] + Density[p]
    return Fitness

Fitness = Fitness_Calculate(Raw_Fitness, Density, Population)

In [ ]:
def Archive_Find(Fitness, S, C, Population):    #Index
    Population_index = list(range(len(Population)))
    Archive = []
    for p in range(len(Population)):
        if Fitness[p] < 1:
            Archive.append(p)
            Population_index.remove(p)

    #Situation 1
    if len(Archive) < Num_Archive: 
        indexed_fitness = list(enumerate(Fitness))
        indexed_fitness.sort(key=lambda x: x[1])  

        for q in range(len(Archive), Num_Archive):   
            Archive.append(indexed_fitness[q][0])
            Population_index.remove(indexed_fitness[q][0])
    
    #Situation 2
    while len(Archive) > Num_Archive:
        distance = np.zeros((len(Archive), len(Archive))) 
        i = 0
        for p in Archive:
            j = 0
            for q in Archive:
                if p == q:
                    distance[i][j] = float('inf')
                else:
                    distance[i][j] = ((S[p] - S[q]) ** 2 + (C[p] - C[q]) ** 2)**(1/2) 
                j += 1
            i += 1
        sorted_distance = np.sort(distance, axis = 1)
        k = math.floor((Num_Chrom + Num_Archive)**(1/2))
        k_distance = sorted_distance[:, k-1]
        min_k_distance = np.min(k_distance)
        min_k_distance_index = np.where(k_distance == min_k_distance)[0]

        if len(min_k_distance_index) == 1:
            Archive.remove(min_k_distance_index)
            Population_index.append(min_k_distance_index)
            
        else:
            unique_distances = np.unique(k_distance)
            for distance in unique_distances[1:]:
                next_min_distance_index = np.where(k_distance == distance)[0]
                if len(next_min_distance_index) == 1:
                    Archive.remove(next_min_distance_index[0])
                    Population_index.append(next_min_distance_index[0])     
    return Archive, Population_index

Archive, Population_index = Archive_Find(Fitness, S, C, Population)

In [ ]:
def Archive_S_C(Archive, S, C): #System reliability and cost of archive
    Archive_S = []
    Archive_C = []
    for i in Archive:
        Archive_S.append(S[i])
        Archive_C.append(C[i])
    return Archive_S, Archive_C

Archive_S, Archive_C = Archive_S_C(Archive, S, C)

In [ ]:
def Archive_Population_Fitness(Archive, Population_index, Fitness, Population, Population_Num):   #Fitness of archive and population and machine configuration of Archive
    Archive_Population = []
    Archive_Population_Num = []
    Archive_Fitness = [] 
    Population_Fitness = [] 
    for i in Archive:
        Archive_Fitness.append(Fitness[i]) 
        Archive_Population.append(Population[i])
        Archive_Population_Num.append(Population_Num[i])
    for i in Population_index:
        Population_Fitness.append(Fitness[i]) 
    return Archive_Fitness, Archive_Population, Archive_Population_Num, Population_Fitness
    
Archive_Fitness, Archive_Population, Archive_Population_Num, Population_Fitness = Archive_Population_Fitness(Archive, Population_index, Fitness, Population, Population_Num)

In [ ]:
def Binary_Selection_with_replacement(Archive, Archive_Fitness):
    # 第一次選擇
    [j, k] = np.random.choice(Num_Chrom, 2, replace=False)  # 任選兩個index
    if Archive_Fitness[j] <= Archive_Fitness[k]:
        Parent_1 = j
    else:
        Parent_1 = k

    # 第二次選擇
    [j, k] = np.random.choice(Num_Chrom, 2, replace=False)  # 任選兩個index
    if Archive_Fitness[j] <= Archive_Fitness[k]:
        Parent_2 = j
    else:
        Parent_2 = k

    # 若 Parent_1 和 Parent_2 一樣，則重新選擇，直到不一樣
    while Parent_1 == Parent_2:
        [j, k] = np.random.choice(Num_Chrom, 2, replace=False)  # 再任選兩個index
        if Archive_Fitness[j] <= Archive_Fitness[k]:
            Parent_2 = j
        else:
            Parent_2 = k

    return Parent_1, Parent_2

In [ ]:
Pc = 0.6              # Crossover rate
Num_Crossover = int(Pc * Num_Chrom /2)

def Crossover_Mask(Population, Population_num):
    Crossover = []
    Crossover_Num = []
    
    for _ in range(Num_Crossover):
        mask = np.random.randint(2, size = (Num_m, Num_n))
        j, k = Binary_Selection_with_replacement(Archive, Archive_Fitness)        
        Parent_1, Parent_2 = copy.deepcopy(Population[j]), copy.deepcopy(Population[k])
        Parent_1_Num, Parent_2_Num = copy.deepcopy(Population_Num[j]), copy.deepcopy(Population_Num[k])
        
        Child_1 = np.tile(Parent_1, 1)
        Child_2 = np.tile(Parent_2, 1)
        Child_1_Num = np.tile(Parent_1_Num, 1)
        Child_2_Num = np.tile(Parent_2_Num, 1)

        for j in range(Num_m):
            for i in range(Num_n):
                if mask[j][i] == 1:
                    Child_1[j][i] = Parent_2[j][i]
                    Child_2[j][i] = Parent_1[j][i]
                    Child_1_Num[j][i] = Parent_2_Num[j][i]
                    Child_2_Num[j][i] = Parent_1_Num[j][i]
        
        Crossover.append(Child_1)
        Crossover.append(Child_2)
        Crossover_Num.append(Child_1_Num)
        Crossover_Num.append(Child_2_Num)    
    return Crossover, Crossover_Num

Crossover, Crossover_Num = Crossover_Mask(Population, Population_Num)

In [ ]:
Pm = 0.025  # Mutation rate, mutation = p(t+1)

def Mutation_point(Crossover, Crossover_Num, New_Max_num, Rema_Max_num):
    Num_Crossover_2 = 2 * Num_Crossover
    Num_Mutation = int(Pm * Num_Chrom * Num_m * Num_n)
    Mutation = copy.deepcopy(Crossover)
    Mutation_Num = copy.deepcopy(Crossover_Num)
    for _ in range(Num_Mutation):
        Mutation_Gene = np.random.randint(Num_Crossover_2)
        j = np.random.randint(Num_m)
        i = np.random.randint(Num_n)
        
        if Mutation[Mutation_Gene][j][i] == 1:
            Mutation[Mutation_Gene][j][i] = 0
            Mutation_Num[Mutation_Gene][j][i] = np.random.randint(1, New_Max_num[i] + 1)
        else:
            Mutation[Mutation_Gene][j][i] = 1
            Mutation_Num[Mutation_Gene][j][i] = np.random.randint(1, Rema_Max_num[i] + 1) 
    
    return Mutation, Mutation_Num

Mutation,  Mutation_Num = Mutation_point(Crossover, Crossover_Num, New_Max_Num, Rema_Max_Num)

In [ ]:
for i in range(Num_Iteration-1):
    Mutation_S = []
    Mutation_C = []
    for Chromosome, Chromosome_num in zip(Mutation, Mutation_Num):
        Result_Reconstruction_Matrix = AMC_Markov_Reconstruction_Transition_Probability()
        Output_prob, E_First_row = AMC_Markov_Calculate(Result_Reconstruction_Matrix)
        F_vectors = Enumerant_F()
        Input = Calculate_Input()
        Find_max_capacity = Find_max_capacity_from_chromosome(Chromosome)
        Lj = Find_Lj_Chromosome(Find_max_capacity, Chromosome_num)
        Result_Input_Capacity = Result_Input_Capacity_Chromosome(Input, Lj)
        Result_Workstation_Input = Flow_Workstation (Result_Input_Capacity, E_First_row, Find_max_capacity, Chromosome_num)
        Min_num_machine = Calculate_Min_num_machine(Result_Workstation_Input, Find_max_capacity, Chromosome)
        Energy_Machine,  Price_Machine, Production, Distance, Emission_Factor = Energy_Data(Chromosome)
        Result_F_Energy_Scope1_2 = Calculate_Result_F_Energy_Scope1_2(Min_num_machine, Find_max_capacity, Energy_Machine, Distance)
        Result_F_Energy_Scope3 = Calculate_Result_F_Energy_Scope3(Result_F_Energy_Scope1_2, Find_max_capacity, Price_Machine, Emission_Factor, Production)
        d_MP_Candidates = Calculate_d_MP_Candidates(Result_F_Energy_Scope3, Chromosome, Find_max_capacity)
        d_MP, corresponding_Result_F_Energy_Scope3 = Calculate_d_MP(d_MP_Candidates, Result_F_Energy_Scope3)
        Prob_workstation = Load_Prob_Data (Chromosome)
        d_MP = np.array(corresponding_Result_F_Energy_Scope3)  # 將 d_MP 轉為陣列型式(機台數量)
        Num_z = d_MP.shape[0]
        System_Reliability = Calculate_System_Reliability(d_MP, Num_z, Prob_workstation, Chromosome_num)
        Mutation_S.append(System_Reliability)
        Cost = Calculate_Cost_Machine(Price_Machine, Chromosome_num)
        Mutation_C.append(Cost)

    Combination_Population = Archive_Population + Mutation
    Combination_Population_num = Archive_Population_Num + Mutation_Num
    Combination_S = Archive_S + Mutation_S
    Combination_C = Archive_C + Mutation_C

    Combination_strength, Combination_index = strength(Combination_S, Combination_C, Combination_Population)
    Combination_Raw_Fitness = Raw_Fitness_Calculate(Combination_strength, Combination_index, Combination_Population)
    Combination_Density = Density_Calculate(Combination_S, Combination_C, Combination_Population)
    Combination_Fitness = Fitness_Calculate(Combination_Raw_Fitness, Combination_Density, Combination_Population)
    Combination_Archive, Combination_Population_index = Archive_Find(Combination_Fitness, Combination_S, Combination_C, Combination_Population)
    Combination_Archive_S, Combination_Archive_C = Archive_S_C(Combination_Archive, Combination_S, Combination_C)
    Combination_Archive_Fitness, Combination_Archive_Population, Combination_Archive_Population_num, Combination_Population_Fitness = Archive_Population_Fitness(Combination_Archive, Combination_Population_index, Combination_Fitness, Combination_Population, Combination_Population_num)        
    Combination_Crossover, Combination_Crossover_Num = Crossover_Mask(Combination_Population, Combination_Population_num)
    Mutation,  Mutation_Num = Mutation_point(Combination_Crossover, Combination_Crossover_Num, New_Max_Num, Rema_Max_Num)

In [ ]:
Mutation_S = []
Mutation_C = []

for Chromosome, Chromosome_num in zip(Mutation, Mutation_Num):
    Result_Reconstruction_Matrix = AMC_Markov_Reconstruction_Transition_Probability()
    Output_prob, E_First_row = AMC_Markov_Calculate(Result_Reconstruction_Matrix)
    F_vectors = Enumerant_F()
    Input = Calculate_Input()
    Find_max_capacity = Find_max_capacity_from_chromosome(Chromosome)
    Lj = Find_Lj_Chromosome(Find_max_capacity, Chromosome_num)
    Result_Input_Capacity = Result_Input_Capacity_Chromosome(Input, Lj)
    Result_Workstation_Input = Flow_Workstation (Result_Input_Capacity, E_First_row, Find_max_capacity, Chromosome_num)
    Min_num_machine = Calculate_Min_num_machine(Result_Workstation_Input, Find_max_capacity, Chromosome)
    Energy_Machine,  Price_Machine, Production, Distance, Emission_Factor = Energy_Data(Chromosome)
    Result_F_Energy_Scope1_2 = Calculate_Result_F_Energy_Scope1_2(Min_num_machine, Find_max_capacity, Energy_Machine, Distance)
    Result_F_Energy_Scope3 = Calculate_Result_F_Energy_Scope3(Result_F_Energy_Scope1_2, Find_max_capacity, Price_Machine, Emission_Factor, Production)
    d_MP_Candidates = Calculate_d_MP_Candidates(Result_F_Energy_Scope3, Chromosome, Find_max_capacity)
    d_MP, corresponding_Result_F_Energy_Scope3 = Calculate_d_MP(d_MP_Candidates, Result_F_Energy_Scope3)
    Prob_workstation = Load_Prob_Data (Chromosome)
    d_MP = np.array(corresponding_Result_F_Energy_Scope3)  # 將 d_MP 轉為陣列型式
    Num_z = d_MP.shape[0]
    System_Reliability = Calculate_System_Reliability(d_MP, Num_z, Prob_workstation, Chromosome_num)
    Mutation_S.append(System_Reliability)
    Cost = Calculate_Cost_Machine(Price_Machine, Chromosome_num)
    Mutation_C.append(Cost)

Combination_Population = Archive_Population + Mutation
Combination_Population_num = Archive_Population_Num + Mutation_Num
Combination_S = Archive_S + Mutation_S
Combination_C = Archive_C + Mutation_C

Combination_strength, Combination_index = strength(Combination_S, Combination_C, Combination_Population)
Combination_Raw_Fitness = Raw_Fitness_Calculate(Combination_strength, Combination_index, Combination_Population)
Combination_Density = Density_Calculate(Combination_S, Combination_C, Combination_Population)
Combination_Fitness = Fitness_Calculate(Combination_Raw_Fitness, Combination_Density, Combination_Population)
Combination_Archive, Combination_Population_index = Archive_Find(Combination_Fitness, Combination_S, Combination_C, Combination_Population)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np

# Your previous Pareto plotting code
def y_axis_formatter(x, pos):
    return f'{x/10000:.0f}'

# Assuming F, All_Population, All_Population_Num, All_Sys, All_Cost are defined elsewhere
Pareto_Chrom = []
Pareto_Chrom_Num = []
Pareto_Sys = []
Pareto_Cost = []

for i in range(len(Combination_Fitness)):  
    if Combination_Fitness[i] < 1:
        Pareto_Chrom.append(Combination_Population[i])
        Pareto_Chrom_Num.append(Combination_Population_num[i])
        Pareto_Sys.append(Combination_S[i])
        Pareto_Cost.append(Combination_C[i])

# Plot the Pareto front
plt.figure(figsize=(10, 6))
plt.scatter(Pareto_Sys, Pareto_Cost, color='b', label='Pareto Solutions')

# Step 1: Decision Matrix
Data_Matrix = np.column_stack((Pareto_Sys, Pareto_Cost))
Weight_Matrix = [0.6, 0.4]
Benefit_attributes = set([0])  # Benefit type (Step 4)
Rows, Cols = Data_Matrix.shape

# Step 2: Normalize
Normalize_Rate = np.zeros(Cols)
Normalize_Matrix = np.zeros((Rows, Cols))
for i in range(Cols):
    for j in range(Rows):
        Normalize_Rate[i] += (Data_Matrix[j][i])**2

for i in range(Cols):
    Normalize_Rate[i] = np.sqrt(Normalize_Rate[i])

for i in range(Cols):
    for j in range(Rows): 
        Normalize_Matrix[j][i] = Data_Matrix[j][i] / Normalize_Rate[i]

# Step 3: Weight
Weighted_Normalize_Matrix = np.zeros((Rows, Cols))
for i in range(Cols):
    for j in range(Rows): 
        Weighted_Normalize_Matrix[j][i] = Normalize_Matrix[j][i] * Weight_Matrix[i]

# Step 4: Positive and Negative Ideal Solutions
a_pos = np.zeros(Cols)
a_neg = np.zeros(Cols)

for i in range(Cols):    
    column = Weighted_Normalize_Matrix[:, i]
    max_val = np.max(column)
    min_val = np.min(column)
    
    if i in Benefit_attributes:
        a_pos[i] = max_val
        a_neg[i] = min_val
    else:
        a_pos[i] = min_val
        a_neg[i] = max_val

# Step 5: Distances
s_pos = np.zeros(Rows)
s_neg = np.zeros(Rows)

for j in range(Rows):
    for i in range(Cols):
        s_pos[j] += (Weighted_Normalize_Matrix[j][i] - a_pos[i])**2
        s_neg[j] += (Weighted_Normalize_Matrix[j][i] - a_neg[i])**2

s_pos = np.sqrt(s_pos)
s_neg = np.sqrt(s_neg)

# Step 6: Ideal Solution
Sol = np.zeros(Rows)
for j in range(Rows):
    Sol[j] = s_neg[j] / (s_pos[j] + s_neg[j])

# Step 7: Ranking
Sorted_indices = np.argsort(Sol)[::-1]

# Get the best solution index (first one after sorting)
best_index = Sorted_indices[0]
best_chrom = Pareto_Chrom[best_index]
best_chrom_num = Pareto_Chrom_Num[best_index]

# Print the best solution details
print(f"Best Chromosome: {best_chrom}")
print(f"Best Chromosome Number: {best_chrom_num}")

# Highlight the best solution on the Pareto plot
plt.scatter(Pareto_Sys[best_index], Pareto_Cost[best_index], color='r', label='Best Solution', zorder=5)

plt.title('Pareto Front')
plt.xlabel('System Reliability')
plt.ylabel('Cost')
plt.gca().yaxis.set_major_formatter(FuncFormatter(y_axis_formatter))

plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# 匯入NSGA-II資料
Pareto_data_1 = pd.read_csv("pareto_1.csv")

# 自定義 y 軸格式
def y_axis_formatter(x, pos):
    return f'{x / 10000:.0f}'

# 繪製圖表
plt.figure(figsize=(10, 6))

# 第一組資料用藍色標示
plt.scatter(Pareto_data_1["System_Reliability"], Pareto_data_1["Cost"], color='b', label='NSGA-II')

# 第二組資料用橘色標示
plt.scatter(Pareto_Sys, Pareto_Cost, color='orange', label='SPEA2')

# 設定軸範圍
plt.xlim(0.2, 1.05)  # 留出右側邊界
plt.ylim(10000050, 79000000)

# 添加標題和軸標籤
plt.title('Pareto Front Comparison')
plt.xlabel('System Reliability')
plt.ylabel('Cost')
plt.gca().yaxis.set_major_formatter(FuncFormatter(y_axis_formatter))

# 添加網格
plt.grid(True)

# 設定圖例位置
plt.legend(loc='upper left', bbox_to_anchor=(0.001, 1))

# 增加圖的邊緣空間
plt.subplots_adjust(left=0.2, right=0.9, top=0.9, bottom=0.1)

# 顯示圖表
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib import rcParams
import numpy as np

# Set font to Times New Roman
rcParams['font.family'] = 'Times New Roman'

def y_axis_formatter(x, pos):
    return f'{x/10000:.0f}'

# Assuming F, All_Population, All_Population_Num, All_Sys, All_Cost are defined elsewhere
Pareto_Chrom = []
Pareto_Chrom_Num = []
Pareto_Sys = []
Pareto_Cost = []

for i in range(len(Combination_Fitness)):  
    if Combination_Fitness[i] < 1:
        Pareto_Chrom.append(Combination_Population[i])
        Pareto_Chrom_Num.append(Combination_Population_num[i])
        Pareto_Sys.append(Combination_S[i])
        Pareto_Cost.append(Combination_C[i])

# Plot the Pareto front
plt.figure(figsize=(10, 6))
plt.scatter(Pareto_Sys, Pareto_Cost, color='b', label='Pareto Solutions')

# Highlight the best solution on the Pareto plot
plt.title('Pareto Front')
plt.xlabel('System Reliability')
plt.ylabel('Cost')
plt.gca().yaxis.set_major_formatter(FuncFormatter(y_axis_formatter))

plt.grid(True)
plt.legend()

# Save the figure
plt.savefig('pareto_front.png', dpi=300, bbox_inches='tight')

# Show the figure
plt.show()


In [ ]:
len(Pareto_Chrom)